In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt


# Parte 1 – Preparação dos dados  
## 1. Compreensão Inicial dos Dados    
### Carregar o Dataset e Visualizar os Dados

In [39]:
movieratings = pd.read_csv('ratings.dat', sep='::', header=None, names=['userId', 'movieId', 'rating', 'timestamp'], engine='python', encoding='latin-1')
movieUser = pd.read_csv('users.dat', sep='::', header=None, names=['userId', 'Gender', 'Age', 'Occupation', 'Zip-code'], engine='python', encoding='latin-1')
movies = pd.read_csv('movies.dat', sep='::', header=None, names=['movieId', 'Title', 'Genres'], engine='python', encoding='latin-1')

**Exibição das 5 primeiras linhas dos conjuntos de dados**

In [40]:
print("5 primeiras linhas do Dataset Movies: ")
movies.head()

5 primeiras linhas do Dataset Movies: 


,movieId,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [41]:
print("5 primeiras linhas do Dataset Movieratings: ")
movieratings.head()

5 primeiras linhas do Dataset Movieratings: 


,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [42]:
print("5 primeiras linhas do Dataset MovieUser: ")
movieUser.head()

5 primeiras linhas do Dataset MovieUser: 


,userId,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


**Descrição das Variáveis – MovieLens 1M**

- ***ratings.dat (Avaliações)***
  - **`userId`** – Identificador único do usuário (1 a 6040)  
  - **`movieId`** – Identificador único do filme (1 a 3952)  
  - **`rating`** – Nota atribuída pelo usuário ao filme, variando de 1 a 5 estrelas (apenas valores inteiros)  
  - **`timestamp`** – Data/hora da avaliação em segundos desde a época Unix (`time(2)`)  
  - **Observações:** Cada usuário possui no mínimo 20 avaliações  
- ***users.dat (Usuários)***
  - **`userId`** – Identificador único do usuário (1 a 6040)  
  - **`Gender`** – Gênero do usuário: `"M"` (masculino) ou `"F"` (feminino)  
  - **`Age`** – Faixa etária do usuário (valores codificados):  
    - 1 – Menos de 18 anos  
    - 18 – 18-24 anos  
    - 25 – 25-34 anos  
    - 35 – 35-44 anos  
    - 45 – 45-49 anos  
    - 50 – 50-55 anos  
    - 56 – 56 anos ou mais  
  - **`Occupation`** – Ocupação (valores codificados):  
    - 0 – Outro ou não especificado  
    - 1 – Acadêmico/Educador  
    - 2 – Artista  
    - 3 – Administrativo/Clerical  
    - 4 – Estudante universitário/pós-graduação  
    - 5 – Atendimento ao cliente  
    - 6 – Médico/Profissional de saúde  
    - 7 – Executivo/Gerencial  
    - 8 – Agricultor  
    - 9 – Do lar  
    - 10 – Estudante de ensino fundamental/médio  
    - 11 – Advogado  
    - 12 – Programador  
    - 13 – Aposentado  
    - 14 – Vendas/Marketing  
    - 15 – Cientista  
    - 16 – Autônomo  
    - 17 – Técnico/Engenheiro  
    - 18 – Artesão/Trabalhador manual  
    - 19 – Desempregado  
    - 20 – Escritor  
  - **`Zip-code`** – CEP informado voluntariamente pelo usuário  
  - **Observações:** As informações demográficas foram fornecidas voluntariamente e não foram verificadas quanto à precisão  

-  ***movies.dat (Filmes)***
  - **`movieId`** – Identificador único do filme (1 a 3952)  
  - **`Title`** – Título do filme conforme consta no IMDb, incluindo o ano de lançamento  
  - **`Genres`** – Lista de gêneros separados por pipe `"|"`. Possíveis gêneros:  
    - Action (Ação)  
    - Adventure (Aventura)  
    - Animation (Animação)  
    - Children's (Infantil)  
    - Comedy (Comédia)  
    - Crime (Crime)  
    - Documentary (Documentário)  
    - Drama (Drama)  
    - Fantasy (Fantasia)  
    - Film-Noir (Film Noir)  
    - Horror (Terror)  
    - Musical (Musical)  
    - Mystery (Mistério)  
    - Romance (Romance)  
    - Sci-Fi (Ficção científica)  
    - Thriller (Suspense)  
    - War (Guerra)  
    - Western (Faroeste)  


**Tamanho do conjunto de dados**

In [43]:
print(f"Tamanho do Dataset Movies: {movies.shape[0]} linhas, {movies.shape[1]} colunas")
print(f"Tamanho do Dataset Movieratings: {movieratings.shape[0]} linhas, {movieratings.shape[1]} colunas")
print(f"Tamanho do Dataset MovieUser: {movieUser.shape[0]} linhas, {movieUser.shape[1]} colunas")


Tamanho do Dataset Movies: 3883 linhas, 3 colunas
Tamanho do Dataset Movieratings: 1000209 linhas, 4 colunas
Tamanho do Dataset MovieUser: 6040 linhas, 5 colunas


**Tipos de variáveis**

In [44]:
print("Tipo de variáveis (Dataset Movies): ")
print(movies.dtypes)

print("\nTipo de variáveis (Dataset Movieratings): ")
print(movieratings.dtypes)

print("\nTipo de variáveis (Dataset MovieUser): ")
print(movieUser.dtypes)

Tipo de variáveis (Dataset Movies): 
movieId     int64
Title      object
Genres     object
dtype: object

Tipo de variáveis (Dataset Movieratings): 
userId       int64
movieId      int64
rating       int64
timestamp    int64
dtype: object

Tipo de variáveis (Dataset MovieUser): 
userId         int64
Gender        object
Age            int64
Occupation     int64
Zip-code      object
dtype: object


**Nome das colunas**

In [45]:
print(f"Nome das colunas (Dataset Movies): {movies.columns.tolist()}")
print(f"Nome das colunas (Dataset Movieratings): {movieratings.columns.tolist()}")
print(f"Nome das colunas (Dataset MovieUser): {movieUser.columns.tolist()}") 

Nome das colunas (Dataset Movies): ['movieId', 'Title', 'Genres']
Nome das colunas (Dataset Movieratings): ['userId', 'movieId', 'rating', 'timestamp']
Nome das colunas (Dataset MovieUser): ['userId', 'Gender', 'Age', 'Occupation', 'Zip-code']


**Valores ausentes**

In [46]:
print("Valores ausentes (Dataset Movies):")
print(movies.isnull().sum())

print("\nValores ausentes (Dataset Movieratings):")
print(movieratings.isnull().sum())

print("\nValores ausentes (Dataset MovieUser):")
print(movieUser.isnull().sum())

Valores ausentes (Dataset Movies):
movieId    0
Title      0
Genres     0
dtype: int64

Valores ausentes (Dataset Movieratings):
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Valores ausentes (Dataset MovieUser):
userId        0
Gender        0
Age           0
Occupation    0
Zip-code      0
dtype: int64


**Ajustar o tipo das variáveis**

In [47]:
#Ajuste nas váriáveis do dataset Movies
movies['movieId'] = movies['movieId'].astype(str)
movies = movies.astype({col: 'category' for col in movies.select_dtypes('object').columns})

#Ajuste nas váriáveis do dataset Movieratings
movieratings['userId'] = movieratings['userId'].astype(str)
movieratings['movieId'] = movieratings['movieId'].astype(str)
movieratings = movieratings.astype({col: 'category' for col in movieratings.select_dtypes('object').columns})

#Ajuste nas váriáveis do dataset MovieUser
movieUser['userId'] = movieUser['userId'].astype(str)
movieUser['Zip-code'] = movieUser['Zip-code'].astype(str)
movieUser = movieUser.astype({col: 'category' for col in movieUser.select_dtypes('object').columns})


print("Tipo de variáveis (Dataset Movies): ")
print(movies.dtypes)

print("\nTipo de variáveis (Dataset Movieratings): ")
print(movieratings.dtypes)

print("\nTipo de variáveis (Dataset MovieUser): ")
print(movieUser.dtypes)

Tipo de variáveis (Dataset Movies): 
movieId    category
Title      category
Genres     category
dtype: object

Tipo de variáveis (Dataset Movieratings): 
userId       category
movieId      category
rating          int64
timestamp       int64
dtype: object

Tipo de variáveis (Dataset MovieUser): 
userId        category
Gender        category
Age              int64
Occupation       int64
Zip-code      category
dtype: object


Ajustes realizados:
 - **Todas as variáveis de identificação (id) foram ajustadas para o tipo string, pois representam um identificador e não, uma variável numérica. O Zip-code (MovieUser) também foi convertido em string (str).**
 - **As variáveis do tipo 'object' foram convertidas para o tipo 'category' para redução no uso de memória e facilidade nos agrupamentos**

### . Disparidade entre Gêneros por Gênero do Usuário



**Combinando os datasets Ratings e Users**

In [48]:
df_merged = movieratings.merge(movieUser, on='userId').merge(movies, on='movieId')
df_merged.head()

,userId,movieId,rating,timestamp,Gender,Age,Occupation,Zip-code,Title,Genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy


**1. Homens e mulheres avaliam gêneros de forma distinta?**

In [53]:
# Criar colunas binárias para cada gênero de filme
genres_split = df_merged['Genres'].str.get_dummies(sep='|')
df = pd.concat([df_merged, genres_split], axis=1)

# Lista com todos os gêneros de filme
genre_cols = genres_split.columns
def mann_whitney_u_test(group1, group2):
    stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
    return stat, p

# Aplicar para cada gênero de filme
resultados = []
for genero in genre_cols:
    aval_masculinas = df[(df['Gender'] == 'M') & (df[genero] == 1)]['rating']
    aval_femininas = df[(df['Gender'] == 'F') & (df[genero] == 1)]['rating']
    stat, p = mann_whitney_u_test(aval_masculinas, aval_femininas)
    resultados.append({'Gênero': genero, 'U': stat, 'p-valor': p})

resultados_df = pd.DataFrame(resultados)
print(resultados_df)

         Gênero             U        p-valor
0        Action  4.828920e+09   6.882910e-01
1     Adventure  1.423297e+09   8.312650e-10
2     Animation  1.815814e+08   1.528616e-13
3    Children's  4.870743e+08  1.564057e-110
4        Comedy  1.209762e+10   8.045700e-61
5         Crime  5.236420e+08   5.121163e-02
6   Documentary  5.736179e+06   5.087702e-01
7         Drama  1.258495e+10   9.110041e-01
8       Fantasy  1.150195e+08   2.430430e-10
9     Film-Noir  3.065314e+07   7.564156e-05
10       Horror  4.536443e+08   4.448463e-01
11      Musical  1.691984e+08   2.370059e-74
12      Mystery  1.482435e+08   1.270611e-02
13      Romance  2.314546e+09   6.053971e-69
14       Sci-Fi  1.790451e+09   9.873567e-02
15     Thriller  2.988691e+09   2.049058e-02
16          War  3.835368e+08   9.870668e-01
17      Western  3.151978e+07   1.895336e-07


***Conclusões***
* Pelos resultados, há diferenças estatisticamente significativas entre homens e mulheres em vários gêneros, mas com tamanhos de efeito (Cliff’s delta) muito pequenos. Ou seja, as diferenças existem, mas são sutis.
* Foram encontradas diferenças estatisticamente significativas (p < 0.05) em vários gêneros, incluindo: Adventure, Animation, Children's, Comedy, Fantasy,  
  Film-Noir, Musical, Romance, Thriller, Western.
* Alguns gêneros não apresentaram diferença significativa: Action, Crime, Documentary, Drama, Horror, Sci-Fi, War.
* No geral, O fator “gênero do filme” tem impacto muito maior nas notas do que o “gênero do usuário”. 


**2. Há interação entre gênero do usuário e do filme?**


In [ ]:
## Teste ANOVA para verificar se a média de avaliações é diferente entre gêneros de filmes para homens e mulheres

estatisticas = []
for genero_filme in genre_cols:
    aval_masculinas = df[(df['Gender'] == 'M') & (df[genero_filme] == 1)]['rating']
    aval_femininas = df[(df['Gender'] == 'F') & (df[genero_filme] == 1)]['rating']
    
    estatisticas.append({
        'Gênero do Filme': genero_filme,
        'Média Masculina': aval_masculinas.mean(),
        'Média Feminina': aval_femininas.mean(),
        'Desvio Padrão Masculino': aval_masculinas.std(),
        'Desvio Padrão Feminino': aval_femininas.std(),
        'n_homens': aval_masculinas.count(),
        'n_mulheres': aval_femininas.count()
    })

# Criar DataFrame com as estatísticas
stats_df = pd.DataFrame(estatisticas)
print("\n--- Médias por gênero de usuário e gênero de filme ---\n")
print(stats_df)

df_long = df.melt(
    id_vars=['userId', 'Gender', 'rating'], 
    value_vars=genre_cols,
    var_name='Gênero do Filme', 
    value_name='PossuiGenero'
)

# Manter apenas registros que realmente pertencem ao gênero
df_long = df_long[df_long['PossuiGenero'] == 1]

modelo = ols('rating ~ C(Gender) * C(Q("Gênero do Filme"))', data=df_long).fit()
tabela_anova = sm.stats.anova_lm(modelo, typ=2)

print("\n--- ANOVA de dois fatores ---\n")
print(tabela_anova)

alpha = 0.05
p_interacao = tabela_anova.loc['C(Gender):C(Q("Gênero do Filme"))', 'PR(>F)']

print("\n--- Resultados: ---")
if p_interacao < alpha:
    print(f"Existe interação significativa entre o gênero do usuário e o gênero do filme (p = {p_interacao:.3e}).")
else:
    print(f"Não foi encontrada interação significativa entre o gênero do usuário e o gênero do filme (p = {p_interacao:.3e}).")



--- Médias por gênero de usuário e gênero de filme ---

   Gênero do Filme  Média Masculina  Média Feminina  Desvio Padrão Masculino  \
0           Action         3.491386        3.490252                 1.131881   
1        Adventure         3.468125        3.512879                 1.129543   
2        Animation         3.661335        3.744702                 1.085677   
3       Children's         3.358961        3.572548                 1.171177   
4           Comedy         3.503667        3.571938                 1.122733   
5            Crime         3.713720        3.689332                 1.073237   
6      Documentary         3.928811        3.946392                 1.033753   
7            Drama         3.766589        3.765662                 1.045872   
8          Fantasy         3.426603        3.513076                 1.134127   
9        Film-Noir         4.092254        4.018087                 0.920489   
10          Horror         3.217891        3.202870            

**Conclusões**  
* Mulheres tendem a avaliarem ligeiramente melhores filmes em infantis (Childen's), romances e musicais. Homens tendem avaliarem com notas mais altas em filmes dos gênero Ocidentais (Western) e Film-noir. Em relação, aos gêneros de filme mais assistidos (Ação, Drama e Comédia), as avaliações são similares.
* C(Gender): F = 440.07, p ≈ 1.07e-97 → diferença média geral entre homens e mulheres significativa, mas o efeito absoluto é pequeno (dado que a diferença média global é pequena).
* C(Gênero do Filme): F = 2206.12, p ≈ 0 → o gênero do filme tem efeito muito forte nas avaliações, como esperado.
* C(Gender):C(Gênero do Filme): F = 69.57, p ≈ 7.65e-241 → interação significativa: o efeito do gênero do usuário depende do gênero do filme.
* A maioria das avaliações dos filmes foram realizados por homens (ex.: em Action, 211.807 vs 45.650), o que pode influenciar as estimativas de média e variância.